# 05 · Casos de uso  ⏱ 20 min

Três situações comuns em que o DuckDB substitui um banco, um cluster ou um script lento. Os dados são sintéticos, gerados no notebook 00.

1. **Sensores industriais:** achar a máquina com defeito, a tendência por hora e o momento do primeiro alarme.
2. **Investigação de logs:** achar o servidor com problema, latência p95, sessões.
3. **Lakehouse local:** Parquet particionado, *pruning* e plano de execução.

In [ ]:
import duckdb
con = duckdb.connect("curso.duckdb")

## Caso 1 · Sensores industriais

A manutenção quer saber se alguma prensa está se comportando diferente das outras. Temos 3 arquivos Parquet com 311 mil leituras de vibração, temperatura e pressão de 12 máquinas, uma leitura a cada 10 segundos.

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW sensores AS
FROM 'dados/sensores/*.parquet'
""")
con.sql("DESCRIBE sensores").show()
con.sql("SELECT linha, count(DISTINCT maquina) AS maquinas, count(*) AS leituras FROM sensores GROUP BY ALL ORDER BY ALL").show()

In [ ]:
# Máquinas com vibração p95 acima do dobro da média da fábrica
con.sql("""
    SELECT maquina, count(*) AS leituras,
           round(avg(vibracao_mm_s), 2)                AS vib_media,
           round(quantile_cont(vibracao_mm_s, 0.95), 2) AS vib_p95
    FROM sensores
    GROUP BY maquina
    QUALIFY vib_p95 > 2 * avg(vib_p95) OVER ()
    ORDER BY vib_p95 DESC
""").show()

In [ ]:
# Tendência por hora: a prensa-07 contra a média das outras
serie = con.sql("""
    SELECT time_bucket(INTERVAL 1 HOUR, ts) AS hora,
           avg(vibracao_mm_s) FILTER (WHERE maquina = 'prensa-07')  AS prensa_07,
           avg(vibracao_mm_s) FILTER (WHERE maquina <> 'prensa-07') AS outras
    FROM sensores GROUP BY ALL ORDER BY hora
""").df()
ax = serie.plot(x="hora", y=["prensa_07", "outras"], figsize=(10, 3), title="Vibração média por hora (mm/s)")

In [ ]:
# A temperatura sobe junto? Correlação por máquina
con.sql("""
    SELECT maquina,
           round(corr(vibracao_mm_s, temperatura_c), 2) AS corr_vib_temp,
           round(max(temperatura_c), 1)                 AS temp_max
    FROM sensores GROUP BY ALL ORDER BY corr_vib_temp DESC LIMIT 4
""").show()

### Quando o alarme deveria ter disparado
Uma leitura isolada alta pode ser ruído. A regra da manutenção aqui é: alarme quando a **média móvel de 10 minutos** passa de 7,1 mm/s. Uma janela com `RANGE` sobre o tempo resolve sem loop.

In [ ]:
con.sql("""
WITH movel AS (
    SELECT maquina, ts,
           avg(vibracao_mm_s) OVER (
               PARTITION BY maquina ORDER BY ts
               RANGE BETWEEN INTERVAL 10 MINUTE PRECEDING AND CURRENT ROW
           ) AS vib_10min
    FROM sensores
)
SELECT maquina,
       min(ts)                               AS primeiro_alarme,
       count(*)                              AS leituras_em_alarme,
       round(max(vib_10min), 2)              AS pico_10min
FROM movel
WHERE vib_10min > 7.1
GROUP BY ALL ORDER BY primeiro_alarme
""").show()

## Caso 2 · Investigação de logs

Recebemos reclamações de erro no checkout. Temos 3 arquivos JSON Lines com 600 mil requisições. Sem subir Elasticsearch, sem pandas.

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW logs AS
FROM read_json('dados/logs/*.jsonl', filename = true)
""")
con.sql("DESCRIBE logs").show()

In [ ]:
# Taxa de erro 5xx por rota e servidor
con.sql("""
    SELECT rota, servidor,
           count(*) AS reqs,
           round(100 * avg((status >= 500)::INT), 2) AS pct_5xx
    FROM logs
    GROUP BY rota, servidor
    QUALIFY pct_5xx > 2 * avg(pct_5xx) OVER ()
    ORDER BY pct_5xx DESC
""").show()

In [ ]:
# Latência: média engana, percentis contam a história
con.sql("""
    SELECT rota,
           round(avg(latencia_ms))                          AS media,
           round(quantile_cont(latencia_ms, 0.50))          AS p50,
           round(quantile_cont(latencia_ms, 0.95))          AS p95,
           round(quantile_cont(latencia_ms, 0.99))          AS p99
    FROM logs GROUP BY ALL ORDER BY p95 DESC
""").show()

In [ ]:
# Erros por hora no servidor suspeito (série temporal)
serie = con.sql("""
    SELECT time_bucket(INTERVAL 1 HOUR, ts::TIMESTAMP) AS hora,
           count(*) FILTER (WHERE status >= 500 AND servidor = 'api-3') AS erros_api3,
           count(*) FILTER (WHERE status >= 500 AND servidor <> 'api-3') / 2 AS erros_outros_media
    FROM logs GROUP BY ALL ORDER BY hora
""").df()
ax = serie.plot(x="hora", y=["erros_api3", "erros_outros_media"], figsize=(10, 3), title="Erros 5xx por hora")

### Sessões de usuário
Uma sessão termina após 30 minutos sem atividade. `lag` + soma acumulada resolve sem loop.

In [ ]:
con.sql("""
WITH ordenado AS (
    SELECT usuario_id, ts::TIMESTAMP AS ts, rota,
           CASE WHEN ts::TIMESTAMP - lag(ts::TIMESTAMP) OVER w > INTERVAL 30 MINUTE
                  OR lag(ts) OVER w IS NULL THEN 1 ELSE 0 END AS nova_sessao
    FROM logs
    WINDOW w AS (PARTITION BY usuario_id ORDER BY ts)
),
sessoes AS (
    SELECT *, sum(nova_sessao) OVER (PARTITION BY usuario_id ORDER BY ts) AS sessao_id
    FROM ordenado
)
SELECT count(DISTINCT (usuario_id, sessao_id))                        AS sessoes,
       round(count(*) / count(DISTINCT (usuario_id, sessao_id)), 2)   AS reqs_por_sessao,
       round(100 * count(DISTINCT (usuario_id, sessao_id)) FILTER (WHERE rota = 'POST /checkout')
             / count(DISTINCT (usuario_id, sessao_id)), 1)             AS pct_sessoes_com_checkout
FROM sessoes
""").show()

## Caso 3 · Lakehouse local com Parquet particionado

Muitas empresas guardam dados em S3/GCS como Parquet particionado por data. O DuckDB consulta isso direto, lendo **só as partições e colunas necessárias**. Aqui usamos a pasta local `dados/lake/`, mas trocar por `s3://…` funciona igual (com a extensão `httpfs`).

In [ ]:
import os
for raiz, pastas, arquivos in sorted(os.walk("dados/lake/vendas"))[:4]:
    print(raiz, arquivos[:2])

In [ ]:
consulta = """
    SELECT canal, round(sum(quantidade * preco_unitario)) AS receita
    FROM read_parquet('dados/lake/vendas/*/*/*.parquet', hive_partitioning = true)
    WHERE ano = 2025 AND mes = 12
    GROUP BY ALL
"""
con.sql(consulta).show()

plano = con.sql("EXPLAIN ANALYZE " + consulta).fetchall()[0][1]
print("\n".join(l for l in plano.splitlines() if any(k in l for k in ("Total Time", "File Filters", "Scanning Files", "Rows", "TABLE_SCAN", "READ_PARQUET")))[:1500])

No arquivo único, o mesmo filtro precisa abrir o arquivo inteiro, embora ainda pule *row groups* pelas estatísticas min/max do Parquet.

In [ ]:
import time
for nome, fonte in [("arquivo único", "'dados/vendas.parquet'"),
                    ("particionado",  "read_parquet('dados/lake/vendas/*/*/*.parquet', hive_partitioning = true)")]:
    filtro = "year(data_venda) = 2025 AND month(data_venda) = 12" if "único" in nome else "ano = 2025 AND mes = 12"
    t0 = time.perf_counter()
    for _ in range(5):
        con.sql(f"SELECT count(*), sum(quantidade) FROM {fonte} WHERE {filtro}").fetchall()
    print(f"{nome:15s} {(time.perf_counter() - t0) / 5 * 1000:7.1f} ms")

### Bônus com internet: NYC Taxi direto da nuvem
```python
con.sql("""
    SELECT hour(tpep_pickup_datetime) AS hora, count(*) AS corridas, round(avg(tip_amount), 2) AS gorjeta
    FROM 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'
    GROUP BY ALL ORDER BY hora
""").show()
```

In [ ]:
con.close()